In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import imageio_ffmpeg

from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

## Add the path to the DiscEvolution directory
sys.path.append('/home/bentobin/code/DiscEvolution')

## import DiscEvolution modules
from DiscEvolution.chemistry import SimpleCOMolAbund
from DiscEvolution.constants import Mearth, Mjup

## Population Synthesis Directory

In [ ]:
MODEL = 'testmodel'
OUTPUT_DIR = f'/home/bentobin/simulations/{MODEL}/output/'
PKL_FILE = f'/home/bentobin/simulations/{MODEL}/df_planets_{MODEL}_snapshots.pkl'
FIG_DIR = f'/home/bentobin/movies/{MODEL}/'
SAVE_FIGS = True

## Times to plot (in Myr)
time_snapshots = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

## Global Plotting Style

In [ ]:
## Plotting imports
from matplotlib.colors import LogNorm

## Animation frame delay (in milliseconds)

interval = 500
fps = 2

## Global plot style

plt.rcParams.update({
    'axes.linewidth': 2.0,
    'axes.labelsize': 14,
    'axes.labelweight': 'normal',
    'xtick.major.width': 2.0,
    'ytick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.minor.width': 1.5,
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12})

## Method to bold the tick labels

def bold_ticks(*axes):
    for ax in axes:
        labels = (ax.get_xticklabels() + ax.get_yticklabels()
            + ax.get_xticklabels(minor = True) + ax.get_yticklabels(minor = True))

        for label in labels:
            label.set_fontweight('bold')

## Method to Load in Data

In [ ]:
## Method to read data from an HDF5 dataset or group

def read_hdf5_array(obj):
    """
    Read data from either an HDF5 dataset or an HDF5 group.
    """

    ## Normal case (the data are stored as one HDF5 dataset)
    if isinstance(obj, h5py.Dataset):
        return np.array(obj)

    ## Ragged case (the data are stored as a group)
    elif isinstance(obj, h5py.Group):
        values = []

        for key in sorted(obj.keys(), key = int):
            values.append(np.array(obj[key]))

        return values

    ## Raise an error if an unexpected HDF5 object is encountered
    else:
        raise TypeError(f"Unexpected HDF5 object type: {type(obj)}")

## Method to load data from an HDF5 file

def load_data(data_dir):
    """
    Load a disc evolution output file (HDF5 only).

    Parameters:
        data_dir: Path to the HDF5 output file.

    Returns:
        data: Dictionary containing the simulation output.
    """

    data = {}

    ## Open the HDF5 file in read-only mode
    with h5py.File(data_dir, "r") as f:

        ## -----------------------
        ## General simulation data
        ## -----------------------

        general_keys = [
            "R",
            "t",
            "disk_Mdot_star",
            "disk_Mass",
            "Tc",
            "Sigc"]

        for key in general_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = np.array(f[key])

        ## -----------------
        ## Scalar attributes
        ## -----------------

        scalar_attributes = [
            "alpha_SS",
            "psi_DW",
            "Mdot",
            "Mdisk",
            "Rd",
            "pla_eff",
            "f_plt",
            "complete",
            "aborted"]

        for key in scalar_attributes:
            ## Only load the attribute if it exists in the file
            if key in f.attrs:
                data[key] = f.attrs[key].item()

        ## Names of the chemical species (only written when chemistry is on)
        if "chem_species" in f.attrs:
            data["chem_species"] = [s.decode() if isinstance(s, bytes) else str(s) for s in f.attrs["chem_species"]]

        ## -------------
        ## Disk profiles
        ## -------------

        disk_keys = [
            "T",
            "time_snap",
            "Sigma_G",
            "Sigma_dust",
            "Sigma_pebbles",
            "Sigma_planetesimals",
            "Vdrift_grains",
            "Vdrift_pebbles",
            "St_grains",
            "St_pebbles",
            "St_planetesimals",
            "e_planetesimals",
            "i_planetesimals",
            "de2_dt",
            "di2_dt",
            "de2_dt_drag",
            "di2_dt_drag",
            "de2_dt_VS_embryo",
            "di2_dt_VS_embryo",
            "de2_dt_VS_pltsml",
            "di2_dt_VS_pltsml",
            "de2_dt_DF",
            "di2_dt_DF",
            "gas_chem",
            "ice_chem",
            "planetesimal_ice_chem"]

        for key in disk_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

        ## -----------
        ## Planet data
        ## -----------

        planet_keys = [
            "Mcs",
            "Mes",
            "Rp",
            "disk_Mdot_p",
            "Mdot_planetesimal",
            "Mdot_pebble_core",
            "Mdot_pebble_env",
            "Mdot_migration",
            "Mdot_gas",
            "M_iso_planetesimal",
            "M_iso_pebble",
            "X_cores",
            "X_envs",
            "f_env",
            "t_form"]

        for key in planet_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

    return data

## Build Planet Dataframe

In [ ]:
## Method to find the output row closest to each time snapshot

def snapshot_rows(t, time_snapshots):
    """
    Find the output row closest in time to each time snapshot.

    Parameters:
        t: Output times of the simulation in years.
        time_snapshots: Snapshot times in Myr.

    Returns:
        rows: Index of the closest output row to each snapshot.
        exists: False for each snapshot that is past the end of the simulation.
    """

    t_snapshots = np.array(time_snapshots) * 1e6

    rows = np.array([np.argmin(np.abs(t - t_snapshot)) for t_snapshot in t_snapshots])

    ## The last output row can fall just short of the end of the simulation, so
    ## a snapshot only counts as past the end if it is over one row spacing beyond it
    exists = t_snapshots <= t[-1] + (t[-1] - t[-2])

    return rows, exists

## Method to pick out the value of a per-planet array at each time snapshot

def snapshot_values(values, rows, exists):
    """
    Pick out the value of a per-planet array at each time snapshot.

    Parameters:
        values: Array for one planet, with time along the last axis.
        rows: Index of the closest output row to each snapshot.
        exists: False for each snapshot that is past the end of the simulation.

    Returns:
        A list with one entry per snapshot, which is NaN if the time does not
        exist (past the end of the simulation or before the planet was inserted).
    """

    snapshots = values[..., rows].astype(float)
    snapshots[..., ~exists] = np.nan

    return list(np.moveaxis(snapshots, -1, 0))

def build_planet_dataframe(output_dir, pkl_file, time_snapshots):
    """
    Loop over every completed HDF5 output file, store the state of each planet at
    every time snapshot (plus its initial position, implant time and the simulation
    metadata) as one row of a dataframe, and save the dataframe as a pickle file.

    Parameters:
        output_dir: Directory containing the .h5 output files.
        pkl_file: Path to save the pickled dataframe to.
        time_snapshots: Snapshot times in Myr.

    Returns:
        df_planets: Dataframe with one row per planet. The masses, position and
            chemistry are lists with one entry per snapshot.
    """

    planet_rows = []
    chem_species = None
    n_files = 0

    ## ----------------------
    ## Main loop over files
    ## ----------------------

    for fname in sorted(os.listdir(output_dir)):

        if not fname.endswith(".h5"):
            continue

        fpath = os.path.join(output_dir, fname)

        try:
            ## Load raw simulation output
            data = load_data(fpath)

            ## Only use simulations that have completed
            if not bool(data.get("complete", False)):
                continue

            ## Skip if no planets
            if "Mcs" not in data or len(data["Mcs"]) == 0:
                continue

            n_files += 1
            chem_species = data.get("chem_species", chem_species)

            ## Simulation metadata from the HDF5 attributes (shared by every planet in this file)
            metadata = {
                "Mdisk": data.get("Mdisk", np.nan),
                "Mdot": data.get("Mdot", np.nan),
                "Rd": data.get("Rd", np.nan),
                "pla_eff": data.get("pla_eff", np.nan),
                "f_plt": data.get("f_plt", np.nan),
                "alpha_SS": data.get("alpha_SS", np.nan),
                "psi_DW": data.get("psi_DW", np.nan),
                "last_t": float(data["t"][-1]),
            }

            Mcs = np.array(data["Mcs"])
            Mes = np.array(data["Mes"])
            Rps = np.array(data["Rp"])
            t_form = np.array(data["t_form"])
            X_cores = np.array(data["X_cores"]) if "X_cores" in data else None
            X_envs = np.array(data["X_envs"]) if "X_envs" in data else None

            ## Output rows closest to each time snapshot
            rows, exists = snapshot_rows(data["t"], time_snapshots)

            ## Loop planets
            for ip in range(len(Mcs)):
                planet_rows.append({
                    ## Identifiers
                    "file": fname,
                    "planet_index": ip,

                    ## Snapshot times
                    "time_snapshots": list(time_snapshots),

                    ## Masses and position at each snapshot
                    "Mc": snapshot_values(Mcs[ip], rows, exists),
                    "Me": snapshot_values(Mes[ip], rows, exists),
                    "Mp": snapshot_values(Mcs[ip] + Mes[ip], rows, exists),
                    "Rp": snapshot_values(Rps[ip], rows, exists),

                    ## Chemistry at each snapshot
                    "X_core": snapshot_values(X_cores[ip], rows, exists) if X_cores is not None else None,
                    "X_env": snapshot_values(X_envs[ip], rows, exists) if X_envs is not None else None,

                    ## Implantation
                    "Rp_init": Rps[ip][~np.isnan(Rps[ip])][0],
                    "t_implant": float(t_form[ip]),

                    ## Simulation metadata
                    **metadata,
                })

        except Exception as e:
            print(f"Error in file {fname}: {e}")

    ## Convert to dataframe and save to pkl file
    df_planets = pd.DataFrame(planet_rows)
    df_planets.attrs["chem_species"] = chem_species

    print("Processed", n_files, "completed files")
    print("Planets:", df_planets.shape)

    df_planets.to_pickle(pkl_file)

    return df_planets

df_planets = build_planet_dataframe(OUTPUT_DIR, PKL_FILE, time_snapshots)

## Bulk Composition Functions

In [ ]:
## Method to load the core and envelope abundances of every planet at a time snapshot

def snapshot_abundances(df_planets, i):
    """
    Load the core and envelope abundances of every planet in the dataframe at a time snapshot.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        core: SimpleCOMolAbund with one column per planet (core abundances).
        env: SimpleCOMolAbund with one column per planet (envelope abundances).
        Mc: Core masses with shape (N_planets,).
        Me: Envelope masses with shape (N_planets,).
    """

    n_planets = len(df_planets)

    core = SimpleCOMolAbund(n_planets)
    env = SimpleCOMolAbund(n_planets)

    ## The species order in the HDF5 files must match SimpleCOMolAbund
    chem_species = df_planets.attrs.get("chem_species")
    
    if chem_species is not None and list(chem_species) != list(core.names):
        raise ValueError(f"Species order {chem_species} does not match SimpleCOMolAbund {list(core.names)}")

    ## Stack the per-planet arrays into shape (N_species, N_planets)
    core.data[:] = np.stack([X_core[i] for X_core in df_planets["X_core"]], axis = 1)
    env.data[:] = np.stack([X_env[i] for X_env in df_planets["X_env"]], axis = 1)

    Mc = np.array([Mc[i] for Mc in df_planets["Mc"]], dtype = float)
    Me = np.array([Me[i] for Me in df_planets["Me"]], dtype = float)

    return core, env, Mc, Me

## Method to calculate the bulk metallicity of each planet at a time snapshot

def bulk_metallicity(df_planets, i):
    """
    Calculate the bulk metallicity of each planet at a time snapshot.
    This is the mass fraction of the whole planet (core + envelope) in heavy elements.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        Z: Bulk metallicity, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of heavy elements in the core and envelope
    M_Z = core.total_abund * Mc + env.total_abund * Me

    Z = M_Z / (Mc + Me)

    return pd.Series(Z, index = df_planets.index)

## Method to calculate the envelope metallicity of each planet at a time snapshot

def envelope_metallicity(df_planets, i):
    """
    Calculate the envelope metallicity of each planet at a time snapshot.
    This is the mass fraction of the envelope in heavy elements (the rest is H/He).

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        Z_env: Envelope metallicity, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    Z_env = env.total_abund

    return pd.Series(Z_env, index = df_planets.index)

## Method to calculate the total metal mass of each planet at a time snapshot

def bulk_metal_mass(df_planets, i):
    """
    Calculate the total metal mass of each planet at a time snapshot.
    This is the mass of heavy elements in the whole planet (core + envelope).

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        M_Z: Total metal mass in Earth masses, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of heavy elements in the core and envelope
    M_Z = core.total_abund * Mc + env.total_abund * Me

    return pd.Series(M_Z, index = df_planets.index)

## Method to calculate the envelope metal mass of each planet at a time snapshot

def envelope_metal_mass(df_planets, i):
    """
    Calculate the metal mass in the envelope of each planet at a time snapshot.
    This is the mass of heavy elements in the envelope (the rest is H/He).

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        M_Z_env: Envelope metal mass in Earth masses, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of heavy elements in the envelope
    M_Z_env = env.total_abund * Me

    return pd.Series(M_Z_env, index = df_planets.index)

## Method to calculate the volatile to refractory ratio of each planet at a time snapshot

def bulk_volatile_refractory_ratio(df_planets, i):
    """
    Calculate the bulk volatile to refractory mass ratio of each planet at a time snapshot.
    Volatiles are CO, CH4, CO2 and H2O and refractories are C-grain and Si-grain.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        VR: Volatile to refractory ratio, as a Series aligned with df_planets.
            NaN wherever there are no refractories.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of volatiles and refractories in the core and envelope
    volatiles = ((core["CO"] + core["CH4"] + core["CO2"] + core["H2O"]) * Mc
        + (env["CO"] + env["CH4"] + env["CO2"] + env["H2O"]) * Me)

    refractories = ((core["C-grain"] + core["Si-grain"]) * Mc
        + (env["C-grain"] + env["Si-grain"]) * Me)

    with np.errstate(invalid = "ignore", divide = "ignore"):
        VR = volatiles / refractories

    VR = np.where(np.isfinite(VR), VR, np.nan)

    return pd.Series(VR, index = df_planets.index)

## Method to calculate the bulk water content of each planet at a time snapshot

def bulk_water_content(df_planets, i):
    """
    Calculate the bulk water content of each planet at a time snapshot.
    This is the mass fraction of the whole planet (core + envelope) that is H2O.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        f_H2O: Water mass fraction, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of water in the core and envelope
    M_H2O = core["H2O"] * Mc + env["H2O"] * Me

    f_H2O = M_H2O / (Mc + Me)

    return pd.Series(f_H2O, index = df_planets.index)

## Method to calculate the envelope water content of each planet at a time snapshot

def envelope_water_content(df_planets, i):
    """
    Calculate the envelope water content of each planet at a time snapshot.
    This is the mass fraction of the envelope that is H2O.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().
        i: Index of the time snapshot.

    Returns:
        f_H2O_env: Envelope water mass fraction, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = snapshot_abundances(df_planets, i)

    ## Mass of water in the envelope
    M_H2O_env = env["H2O"] * Me

    f_H2O_env = M_H2O_env / Me

    return pd.Series(f_H2O_env, index = df_planets.index)

## Animate Bulk Compositions

In [ ]:
## Shared setup for the bulk composition movies below
## (each plot is against total planet mass, coloured by current position)

## Solar metallicity used for the envelope metallicity in solar units
Z_solar = 0.02

## Number of frames in each movie
n_times = len(time_snapshots)

## Total planet mass in Jupiter masses (the saved masses are in Earth masses)
## Both arrays have shape (N_planets, N_times)
Mp = np.array(df_planets["Mp"].tolist()) * Mearth / Mjup
Rp = np.array(df_planets["Rp"].tolist())

## Colour: current position on a log scale with the RdYlBu colormap
## The limits span every time snapshot so that the colours mean the same thing in each frame
cmap = plt.cm.RdYlBu
norm = LogNorm(vmin = np.nanmin(Rp), vmax = np.nanmax(Rp))

## Method to animate a quantity against total planet mass over the time snapshots

def animate_population(fig, ax, quantity, filename):
    """
    Animate a scatter plot of a quantity against total planet mass, coloured by
    current position, over the time snapshots and save it as a .gif and .mp4.

    Parameters:
        fig, ax: Figure and axes, with the limits and labels already set.
        quantity: Function of (df_planets, i) giving the y values at snapshot i.
        filename: Name of the saved movies, without the file extension.

    Returns:
        ani: The animation.
    """

    y = [np.asarray(quantity(df_planets, i), dtype = float) for i in range(n_times)]

    scatter = ax.scatter(Mp[:, 0], y[0], c = Rp[:, 0], cmap = cmap, norm = norm, s = 30,
                         edgecolors = "white", linewidths = 0.5, zorder = 3)

    time_text = ax.text(0.95, 0.05, '', transform = ax.transAxes,
        ha = 'right', va = 'bottom', fontsize = 14, fontweight = 'bold', zorder = 4,
        bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8, edgecolor = '0.8'))

    ## Colourbar for current position
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm = norm, cmap = cmap), ax = ax)
    cbar.set_label(r"Position [AU]")

    fig.tight_layout()

    ## Bold the tick labels on the plot and the colourbar
    bold_ticks(ax, cbar.ax)

    ## Update the animation for frame i

    def update(i):
        scatter.set_offsets(np.column_stack([Mp[:, i], y[i]]))
        scatter.set_array(Rp[:, i])
        time_text.set_text(f't = {time_snapshots[i]} Myr')

        return scatter, time_text

    ani = FuncAnimation(fig, update, frames = n_times, interval = interval, blit = False)

    ## Save the movies to FIG_DIR
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok = True)

        gif_path = os.path.join(FIG_DIR, f"{filename}.gif")
        ani.save(gif_path, writer = PillowWriter(fps = fps))

        mp4_path = os.path.join(FIG_DIR, f"{filename}.mp4")
        ani.save(mp4_path, writer = FFMpegWriter(fps = fps))

    plt.close(fig)

    return ani

In [ ]:
## Total metal mass of each planet against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

## Thorngren et al. (2016) mass-metallicity fit
mass_range = np.logspace(-3, 2, 100)
thorngren_metal_mass = 9.7 * mass_range * mass_range ** (-0.45) * 0.02 * 317.8

## Uncertainty region: the lowest and highest values over every combination of coefficient and exponent at each mass
thorngren_bounds = [coefficient * mass_range * mass_range ** exponent * 0.02 * 317.8
                    for coefficient in (9.7 - 1.28, 9.7 + 1.28) for exponent in (-0.45 - 0.09, -0.45 + 0.09)]
thorngren_lower = np.min(thorngren_bounds, axis = 0)
thorngren_upper = np.max(thorngren_bounds, axis = 0)

## Uncertainty region shaded in light gray and bounded by dashed lines
ax.fill_between(mass_range, thorngren_lower, thorngren_upper, color = 'lightgray', alpha = 0.5, zorder = 0)
ax.plot(mass_range, thorngren_metal_mass, 'k-', linewidth = 2, label = 'Thorngren et al. (2016)', zorder = 2)
ax.plot(mass_range, thorngren_lower, 'k--', linewidth = 2, zorder = 2)
ax.plot(mass_range, thorngren_upper, 'k--', linewidth = 2, zorder = 2)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-1, 1e3)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel(r"Total Metal Mass [M$_\oplus$]")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ax.legend(loc = 'upper left')

ani = animate_population(fig, ax, bulk_metal_mass, "total_metal_mass")

HTML(ani.to_jshtml())

In [ ]:
## Envelope metal mass of each planet against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-3, 1e3)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel(r"Envelope Metal Mass [M$_\oplus$]")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ani = animate_population(fig, ax, envelope_metal_mass, "envelope_metal_mass")

HTML(ani.to_jshtml())

In [ ]:
## Envelope metallicity of each planet (in solar units) against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

## Welbanks et al. (2019) solar system trend line based on CH4
mass_range = np.logspace(-3, 2, 100)
welbanks_metallicity = 10 ** (-1.11 * np.log10(mass_range) + 0.38)

## Uncertainty region: the lowest and highest values over every combination of slope and intercept at each mass
welbanks_bounds = [10 ** (slope * np.log10(mass_range) + intercept)
                   for slope in (-1.11 - 0.10, -1.11 + 0.11) for intercept in (0.38 - 0.06, 0.38 + 0.06)]
welbanks_lower = np.min(welbanks_bounds, axis = 0)
welbanks_upper = np.max(welbanks_bounds, axis = 0)

## Uncertainty region shaded in light gray and bounded by dashed lines
ax.fill_between(mass_range, welbanks_lower, welbanks_upper, color = 'lightgray', alpha = 0.5, zorder = 0)
ax.plot(mass_range, welbanks_metallicity, 'k-', linewidth = 2, label = r'Solar System CH$_4$/H (Welbanks et al. 2019)', zorder = 2)
ax.plot(mass_range, welbanks_lower, 'k--', linewidth = 2, zorder = 2)
ax.plot(mass_range, welbanks_upper, 'k--', linewidth = 2, zorder = 2)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-2, 1e3)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel(r"Envelope Metallicity [Z$_\odot$]")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ax.legend(loc = 'upper right')

ani = animate_population(fig, ax, lambda df_planets, i: envelope_metallicity(df_planets, i) / Z_solar, "envelope_metallicity")

HTML(ani.to_jshtml())

In [ ]:
## Bulk volatile to refractory ratio of each planet against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-5, 1e2)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel("Volatile / Refractory Ratio")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ani = animate_population(fig, ax, bulk_volatile_refractory_ratio, "volatile_refractory_ratio")

HTML(ani.to_jshtml())

In [ ]:
## Bulk water content of each planet against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-6, 1e0)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel(r"Bulk Water Content [M$_{\rm H_2O}$/M$_{\rm p}$]")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ani = animate_population(fig, ax, bulk_water_content, "bulk_water_content")

HTML(ani.to_jshtml())

In [ ]:
## Calculate the solar water mass fraction for comparison with the bulk water content of the planets

from DiscEvolution.chemistry import SimpleCOAtomAbund

## Water mass fraction for solar composition with all of the oxygen in H2O
X_solar = SimpleCOAtomAbund(1)
X_solar.set_solar_abundances()
H2O_solar = 18 * X_solar.number_abund("O")[0]

In [ ]:
## Envelope water content of each planet against its total mass

fig, ax = plt.subplots(figsize = (8, 6))

## Welbanks et al. (2019) exoplanet H2O trend line, rescaled to a water mass fraction using the solar value
mass_range = np.logspace(-3, 2, 100)
welbanks_water = H2O_solar * 10 ** (-1.09 * np.log10(mass_range) - 0.95)

## Uncertainty region: the lowest and highest values over every combination of slope and intercept at each mass
welbanks_bounds = [H2O_solar * 10 ** (slope * np.log10(mass_range) + intercept)
                   for slope in (-1.09 - 0.33, -1.09 + 0.34) for intercept in (-0.95 - 0.19, -0.95 + 0.21)]
welbanks_lower = np.min(welbanks_bounds, axis = 0)
welbanks_upper = np.max(welbanks_bounds, axis = 0)

## Uncertainty region shaded in light gray and bounded by dashed lines
ax.fill_between(mass_range, welbanks_lower, welbanks_upper, color = 'lightgray', alpha = 0.5, zorder = 0)
ax.plot(mass_range, welbanks_water, 'k-', linewidth = 2, label = r'Exoplanet H$_2$O (Welbanks et al. 2019)', zorder = 2)
ax.plot(mass_range, welbanks_lower, 'k--', linewidth = 2, zorder = 2)
ax.plot(mass_range, welbanks_upper, 'k--', linewidth = 2, zorder = 2)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1e-3, 1e2)
ax.set_ylim(1e-6, 1e0)

ax.set_xlabel(r"Total Planet Mass [M$_{\rm J}$]")
ax.set_ylabel(r"Envelope Water Content [M$_{\rm H_2O}$/M$_{\rm env}$]")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

ax.legend(loc = 'upper right')

ani = animate_population(fig, ax, envelope_water_content, "envelope_water_content")

HTML(ani.to_jshtml())